# 2.1 선형회귀: 모델, 비용함수, 경사하강법 — 실습 노트북

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/ml1/chapter02_1_linear_regression_gd.ipynb)

책 본문: [2.1 선형회귀: 모델, 비용함수, 경사하강법](https://smhanlab.com/book-ml/kor/ml1/chapter02/1.html)

이 노트북은 책 2.1절의 내용을 코드로 재현합니다: (1) 본문의 순수 Python `gradient_descent`를 실행해 5스텝 표와 발산 출력을 재현, (2) 그래디언트 수식 \(\frac{1}{m}\sum(h-y)\cdot x\)를 **수치미분과 비교**로 검증, (3) 학습률 3개를 300개 데이터 위에서 경사시켜 **"10배 규칙"**을 확인, (4) \(J(w)\)의 1차원 단면과 2차원 등고선 위에 경사하강법 경로를 시각화(본문에 `ch02_gradient_descent_1d.svg`, `ch02_cost_landscape_2d.svg`로 삽입), (5) diabetes 실데이터에서 **학습 데이터 크기가 수렴 속도와 RMSE에 미치는 영향**을 측정(`ch02_diabetes_learning_curve.svg`). numpy/matplotlib/sklearn만 씁니다.

## 0. 설정: 한글 폰트와 import

그래프의 한글 라벨이 깨지지 않도록 CJK 폰트를 골라둡니다 (Colab에 기본 설치).

In [1]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib import font_manager

for _f in ["Noto Sans CJK KR", "NanumGothic", "Malgun Gothic", "AppleGothic"]:
    if any(_f.lower() == x.name.lower() for x in font_manager.fontManager.ttflist):
        plt.rcParams["font.sans-serif"] = [_f, "DejaVu Sans"]
        break
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["svg.fonttype"] = "path"  # SVG로 내보낼 때 글자를 path로 변환
IMG = "/home/smhan/book-ml/kor/src/images"

## 1. 본문의 `gradient_descent`를 실행: 5스텝 표 재현

본문 '손으로 한 번'의 데이터셋 \(X=[1,2,3]\), \(y=[3,5,7]\)(참값 \(y=2x+1\)), 초기값 \(w_0=w_1=0\), 학습률 \(\alpha=0.1\)으로 5스텝을 돌리면 본문의 표와 정확히 일치해야 합니다.

In [2]:
def gradient_descent(X, y, alpha, epochs):
    m, n = len(X), len(X[0])
    w = [0.0] * (n + 1)  # w[0]=bias
    for _ in range(epochs):
        grad = [0.0] * (n + 1)
        for i in range(m):
            pred = w[0] + sum(w[j+1] * X[i][j] for j in range(n))
            error = pred - y[i]
            grad[0] += error
            for j in range(n):
                grad[j+1] += error * X[i][j]
        for j in range(n + 1):
            w[j] -= alpha * grad[j] / m
    return w

# 5스텝 표를 재현하려면 매 스텝 이후 (w0, w1, J)을 기록하는 버전을 쓴다
X = [[1.0], [2.0], [3.0]]; y = [3.0, 5.0, 7.0]; m = 3
w0, w1 = 0.0, 0.0
J0 = sum((w0 + w1 * xi - yi) ** 2 for xi, yi in zip([x[0] for x in X], y)) / (2 * m)
print(f"step 0(초기): w0={w0:.4f} w1={w1:.4f} J={J0:.4f}")
for s in range(1, 6):
    err = [w0 + w1 * xi - yi for xi, yi in zip([x[0] for x in X], y)]
    w0 -= 0.1 * sum(err) / m
    w1 -= 0.1 * sum(e * xi for e, xi in zip(err, [x[0] for x in X])) / m
    err = [w0 + w1 * xi - yi for xi, yi in zip([x[0] for x in X], y)]
    J = sum(e * e for e in err) / (2 * m)
    print(f"step {s}: w0={w0:.4f} w1={w1:.4f} J={J:.4f}")

# 본문 코드 그대로 1000스텝: 참값 (1, 2)에 수렴
print("본문 함수 1000스텝:", gradient_descent(X, y, alpha=0.1, epochs=1000))

step 0(초기): w0=0.0000 w1=0.0000 J=13.8333
step 1: w0=0.5000 w1=1.1333 J=2.7443
step 2: w0=0.7233 w1=1.6378 J=0.5448
step 3: w0=0.8234 w1=1.8621 J=0.1086
step 4: w0=0.8687 w1=1.9618 J=0.0221
step 5: w0=0.8894 w1=2.0059 J=0.0049
본문 함수 1000스텝: [0.9999994358835982, 2.0000002481557733]


출력이 본문의 표와 일치한다 — '경사의 반대 방향으로 이동한다'는 말이 실제로 숫자 위에서 어떻게 읽히는지 보여주는 것이 이 표의 전부다. 참값 \((w_0, w_1) = (1, 2)\)에 5스텝 만에 \((0.89, 2.01)\)까지 접근한다.

## 2. 그래디언트 수식 검증: 해석미분 대 수치미분

본문의 수식 \(\frac{\partial J}{\partial w_j} = \frac{1}{m}\sum_i (h_w(x^{(i)}) - y^{(i)})\, x_j^{(i)}\)가 실제로 \(J\)의 미분인지, **수치미분**(중심차분 \(\frac{J(\theta_+ h) - J(\theta_- h)}{2h}\))으로 확인한다. 임의의 \(w\)에서 두 값이 맞아야 "수식이 옳다"는 뜻이다. 이 방법은 나중에 시그모이드 미분(2.3절)에서 그대로 재사용된다.

In [3]:
Xr = np.array([1.0, 2.0, 3.0]); yr = np.array([3.0, 5.0, 7.0])

def J(w0, w1):
    return 0.5 / len(Xr) * ((w0 + w1 * Xr - yr) ** 2).sum()

def grad_analytic(w0, w1):
    err = (w0 + w1 * Xr) - yr
    return np.array([err.mean(), (err * Xr).mean()])

def grad_numeric(w0, w1, h=1e-5):
    return np.array([(J(w0 + h, w1) - J(w0 - h, w1)) / (2 * h),
                     (J(w0, w1 + h) - J(w0, w1 - h)) / (2 * h)])

for (a, b) in [(0.0, 0.0), (0.5, 1.2), (-1.3, 3.7)]:
    ga, gn = grad_analytic(a, b), grad_numeric(a, b)
    print(f"w=({a}, {b}):  해석 {ga.round(6)}   수치 {gn.round(6)}   최대 오차 {np.abs(ga - gn).max():.2e}")

# 여러 임의 점에서 오차가 항상 ~1e-9 이하인지 확인
rng = np.random.default_rng(0)
worst = max(np.abs(grad_analytic(*p) - grad_numeric(*p)).max() for p in rng.uniform(-3, 3, size=(50, 2)))
print(f"50개 임의 점에서 최대 오차: {worst:.2e}  (중심차분 오차 한계 h^2/6 ~ {1e-5**2/6:.1e})")
assert worst < 1e-6, "해석미분과 수치미분이 맞지 않음!"

w=(0.0, 0.0):  해석 [ -5.       -11.333333]   수치 [ -5.       -11.333333]   최대 오차 1.92e-10
w=(0.5, 1.2):  해석 [-2.1      -4.733333]   수치 [-2.1      -4.733333]   최대 오차 3.95e-11
w=(-1.3, 3.7):  해석 [1.1      3.333333]   수치 [1.1      3.333333]   최대 오차 4.27e-11
50개 임의 점에서 최대 오차: 1.15e-09  (중심차분 오차 한계 h^2/6 ~ 1.7e-11)


해석미분(수식)과 수치미분이 모든 점에서 일치한다 — **수식은 \(J\)를 실제로 미분한 것**이라는 확인이다. "수식대로 코드를 썼다"가 아니라 "수식 자체가 맞다"를 보여주는 것이 검증의 목적이다.

## 3. 학습률의 함정: 수렴 vs 발산 한 장에

본문 '자주 하는 실수'의 \(\alpha=1.5\) 발산 출력을 재현하고, 같은 데이터에서 **발산 경계값**을 계산한다. \(J(w)\)가 이차함수이므로, \(w \leftarrow w - \alpha \nabla J\)는 매 스텝마다 고유방향별로 \((1 - \alpha \lambda_i)\)배 오차를 곱하는 선형 연산이다. 어떤 방향이든 줄려면 \(|1 - \alpha \lambda_{\max}| < 1\), 즉 \(0 < \alpha < 2/\lambda_{\max}\)여야 한다 — 경계값 \(\alpha^* = 2/\lambda_{\max}\)를 \(X^TX/m\)의 가장 큰 고유값으로 직접 계산한다.

In [4]:
# (1) alpha=1.5 발산 재현 (본문 코드 그대로)
w0, w1 = 0.0, 0.0
for i in range(1, 6):
    err = (w0 + w1 * Xr) - yr
    g0, g1 = err.mean(), (err * Xr).mean()
    w0 -= 1.5 * g0
    w1 -= 1.5 * g1
    print(f"step {i}: w0={w0:.4f}   w1={w1:.4f}   cost={J(w0, w1):.4f}")

# (2) 발산 경계값
A = np.column_stack([np.ones(3), Xr]).T @ np.column_stack([np.ones(3), Xr]) / 3
lam_max = np.linalg.eigvalsh(A).max()
alpha_star = 2 / lam_max
print(f"\nA = X^TX/m의 고유값: {np.linalg.eigvalsh(A).round(4)}")
print(f"발산 경계 alpha* = 2/lam_max = {alpha_star:.4f}")
print("-> 0.1(수렴), 0.5(> 0.3606 -> 발산), 1.5(발산)의 운명이 이 숫자로 정해진다")
assert 0.3 < alpha_star < 0.4

step 1: w0=7.5000   w1=17.0000   cost=741.1250
step 2: w0=-47.2500   w1=-107.5000   cost=39708.0312
step 3: w0=353.6250   w1=803.7500   cost=2127480.1953
step 4: w0=-2580.5625   w1=-5866.3750   cost=113986311.4707
step 5: w0=18896.9062   w1=42956.9375   cost=6107168110.2544

A = X^TX/m의 고유값: [0.1202 5.5465]
발산 경계 alpha* = 2/lam_max = 0.3606
-> 0.1(수렴), 0.5(> 0.3606 -> 발산), 1.5(발산)의 운명이 이 숫자로 정해진다


경계값 \(\alpha^* \approx 0.3606\)이 확인 문제의 답이다: **\(\alpha = 0.5\)는 수렴하지 않는다** — 발산이 시작되는 경계는 데이터의 스케일(여기서는 \(X^TX\)의 고유값)에 의해 정해진다. 이 예는 "학습률을 10배씩 줄여가며 찾자"는 본문 조언이 임의의 경험칙이 아니라, 데이터 특이적인 경계값이 존재하기 때문임을 보여준다.

## 4. 학습 곡선: 300개 데이터에서 "10배 규칙"

작은 장난감 데이터(3점)로는 학습률 비교가 발산/비발산의 이분법으로만 보이지만, 실사용 크기의 데이터에서는 **느린 수렴**이 더 흔한 실패모드다. \(y = 3x + 1 + \text{노이즈}(0, 0.5^2)\), 300개 데이터에서 \(\alpha \in \{0.001, 0.01, 0.1\}\)으로 경사하강법을 돌리고, 비용이 **소음 바닥**(최소제곱 해의 \(J\) — 노이즈가 만들어낸 줄 수 없는 최소 비용)에 도달하는 데 걸린 스텝을 센다.

In [5]:
rng = np.random.default_rng(42)
xb = rng.normal(0, 1, 300); yb = 3 * xb + 1 + rng.normal(0, 0.5, 300); mb = 300

# 소음 바닥: 정규방정식 해에서의 J (이 데이터의 이론적 최소)
Abig = np.column_stack([np.ones(mb), xb])
w_star = np.linalg.solve(Abig.T @ Abig, Abig.T @ yb)
J_floor = 0.5 / mb * ((Abig @ w_star - yb) ** 2).sum()
print(f"w* = {w_star.round(4)},  J_floor = {J_floor:.4f} (노이즈가 만든 불가피한 최소 비용)")

def run(alpha, max_steps=3000):
    w0 = w1 = 0.0; J_hist = []; hit = None
    for s in range(1, max_steps + 1):
        err = (w0 + w1 * xb) - yb
        w0 -= alpha * err.mean()
        w1 -= alpha * (err * xb).mean()
        J = 0.5 / mb * ((w0 + w1 * xb - yb) ** 2).sum()
        J_hist.append(J)
        if hit is None and (J - J_floor) < 1e-3:
            hit = s
    return J_hist, hit

results = {}
for a in (0.001, 0.01, 0.1):
    Jh, hit = run(a)
    results[a] = (Jh, hit)
    print(f"alpha={a}: J[1]={Jh[0]:.4f}  J[100]={Jh[99]:.4f}  J[1000]={Jh[999]:.4f}  -> 바닥 도달 스텝: {hit}")

print(f"\n10배 규칙: alpha 0.01->0.1은 {results[0.01][1]} -> {results[0.1][1]} 스텝, 약 {results[0.01][1] / results[0.1][1]:.0f}배 빨라짐")
print("alpha=0.001은 3000스텝 내 도달 불가 (최종 J = {:.4f})".format(results[0.001][0][-1]))

w* = [0.9951 3.0178],  J_floor = 0.1287 (노이즈가 만든 불가피한 최소 비용)
alpha=0.001: J[1]=4.4283  J[100]=3.7598  J[1000]=0.9101  -> 바닥 도달 스텝: None
alpha=0.01: J[1]=4.3625  J[100]=0.9049  J[1000]=0.1287  -> 바닥 도달 스텝: 489
alpha=0.1: J[1]=3.7321  J[100]=0.1287  J[1000]=0.1287  -> 바닥 도달 스텝: 47

10배 규칙: alpha 0.01->0.1은 489 -> 47 스텝, 약 10배 빨라짐
alpha=0.001은 3000스텝 내 도달 불가 (최종 J = 0.1544)


본문 1D 단면 그림의 숫자가 여기에서 나온다: \(\alpha=0.01\)이 약 489스텝, \(\alpha=0.1\)이 약 47스텝이면 학습률을 10배 올리면 수렴이 **약 10배** 빨라진다. \(\alpha=0.001\)은 3000스텝 내 도달 못 한다(최종 \(J=0.154\), 바닥 0.129에서 아직 20% 남음). 참고로 이 데이터의 발산 경계는 \(\alpha^* = 2/\lambda_{\max} \approx 1.98\)이라 \(\alpha=0.1\)까지 올리는 데 발산 위험이 없었다 — §3의 장난감 데이터(\(\alpha^* \approx 0.36\))와 비교하면, **안전한 학습률의 범위는 데이터마다 다르다**는 것의 또 다른 예다.

## 5. 그림 1: \(J(w_1)\) 단면 위에서 경사하강법 걷기 (본문 `ch02_gradient_descent_1d.svg`)

\(w_0\)를 고정하고(=1, 참값 근처) \(w_1\)에만 경사하강법을 적용한 **1차원 단면**이다. \(J(w_1)\)가 포물선(이차함수)인 것을 확인하고, 세 학습률의 경로를 겹쳐서 '한 걸음의 크기'가 어떻게 수렴 속도를 정하는지 한 눈에 보인다.

In [6]:
# w0 = 1 (참값) 고정, w1만 경사하강 — 1차원 단면
w1_grid = np.linspace(-1, 5, 400)
J_curve = 0.5 / mb * ((1.0 + w1_grid[:, None] * xb - yb) ** 2).sum(axis=1)

fig, ax = plt.subplots(figsize=(7, 4.6))
ax.plot(w1_grid, J_curve, "k-", lw=2, label="$J(w_1)$ cross-section ($w_0$=1 fixed)")
for a, c, mk in ((0.001, "C0", "o"), (0.01, "C1", "s"), (0.1, "C2", "^")):
    w1 = 0.0; pts = [0.0]
    for s in range(300):
        err = (1.0 + w1 * xb) - yb
        w1 -= a * (err * xb).mean()
        pts.append(w1)
    ax.plot(range(len(pts)), [J(1.0, p) for p in pts], color=c, marker=mk, ms=3,
            mfc="none", lw=1, label=f"Gradient descent \u03b1={a} ($w_1$ per step)")
ax.axvline(3.0, color="gray", ls=":")
ax.plot([3.0], [J_floor], "k*", ms=14, label="Minimum $w_1^*$\u22483.0")
ax.set_xlabel("Step"); ax.set_ylabel("$J$")
ax.set_title("On the same parabola, step size (learning rate) sets the convergence speed")
ax.legend(fontsize=8, loc="upper right")
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(f"{IMG}/ch02_gradient_descent_1d.svg")
print("ch02_gradient_descent_1d.svg 저장됨")
plt.show()

ch02_gradient_descent_1d.svg 저장됨


세 곡선이 같은 포물선 위에서 출발한다. \(\alpha=0.01\)은 천천히, \(\alpha=0.1\)은 약 10배 빨리 바닥에 닿고, \(\alpha=0.001\)은 300스텝 후에도 거의 움직이지 못했다. **학습률은 '가는 방향'이 아니라 '걸음 수'를 정하는 하이퍼파라미터**라는 본문 주제의 시각화다.

## 6. 그림 2: 2차원 \(J(w)\) 지형과 경사하강법 경로 (본문 `ch02_cost_landscape_2d.svg`)

\((w_0, w_1)\) 평면의 \(J(w)\) 등고선 위에, \((0,0)\)에서 출발한 \(\alpha=0.1\) 경사하강법 경로를 겹친다. 등고선이 **늘어난 타원**(조건수 \(\kappa\))이라는 사실과, 경로가 직선이 아니라 **짧은 축 방향으로 진동(zigzag)하며** 내려간다는 것이 한 번에 보인다 — 2.2절에서 이 "늘어짐"을 조건수로 정량화한다.

In [7]:
w0g = np.linspace(-4, 6, 260); w1g = np.linspace(-4, 8, 260)
W0, W1 = np.meshgrid(w0g, w1g, indexing="ij")
J2 = 0.5 / mb * ((W0[..., None] + W1[..., None] * xb - yb) ** 2).sum(axis=-1)

# alpha=0.1, (0,0) 출발, 600스텝 경로
w0, w1 = 0.0, 0.0; path = [(0.0, 0.0)]
for _ in range(600):
    err = (w0 + w1 * xb) - yb
    w0 -= 0.1 * err.mean()
    w1 -= 0.1 * (err * xb).mean()
    path.append((w0, w1))

fig, ax = plt.subplots(figsize=(9.5, 4.7))
cs = ax.contour(W0, W1, np.log10(J2 + 1e-9), levels=16, cmap="viridis")
ax.clabel(cs, inline=True, fontsize=7, fmt=lambda v: f"{10**v:.3g}")
ax.plot([p[0] for p in path], [p[1] for p in path], "r-", lw=1.6, label="Gradient descent (\u03b1=0.1, 600 steps)")
ax.plot([0.0], [0.0], "rv", ms=10, label="Start (0,0)")
ax.plot([w_star[0]], [w_star[1]], "k*", ms=15, label=f"Minimum w*\u2248({w_star[0]:.2f}, {w_star[1]:.2f})")
ax.set_xlabel("$w_0$ (intercept)"); ax.set_ylabel("$w_1$ (slope)")
ax.set_title("$J(w)$ contours for 300 data points -- elongated ellipses and a zigzag path")
ax.legend(loc="upper right", fontsize=8)
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(f"{IMG}/ch02_cost_landscape_2d.svg")
print("ch02_cost_landscape_2d.svg 저장됨  (최종 위치:", np.round(path[-1], 3), ")")
plt.show()

ch02_cost_landscape_2d.svg 저장됨  (최종 위치: [0.995 3.018] )


경로가 최솟값으로 **일직선**을 타지 않고 짧은 축을 수직으로 오가며(진동) 내려가는 모습은, 등고선이 타원일 때 경사 방향이 항상 "가장 가파른 하강"이되 타원의 긴 축과 대각으로 교차하기 때문이다. 조건수가 1에 가까우면(둥근 등고선) 이 진동은 사라지고 직선에 가깝다 — 2.2절 "조건수" 섹션의 예고편.

## 7. 실데이터: diabetes 데이터셋과 학습 곡선 (본문 `ch02_diabetes_learning_curve.svg`)
이 절에서 계속 3개 데이터로 연습했지만, 실전 문제(특징 10개, 442명)에서 경사하강법이 어떤 모양으로 수렴하는지 확인한다. sklearn의 `load_diabetes`(당뇨 진행도 예측, 10개 신체 특징)의 전체 442명을 쓰고, 모든 특징을 표준화한 뒤(2.2절 스케일링 섹션의 동기 — 표준화하면 \(\lambda_{\max} \approx 4.0\), 경계 \(\alpha^* = 2/\lambda_{\max} \approx 0.50\)이므로 \(\alpha=0.05\)는 안정), 두 가지를 재본다: (a) **같은 실데이터에서 학습률 3개의 비용 곡선** — §4의 10배 규칙이 실데이터에서도 성립하는지, (b) **학습 데이터 크기 \(n\)을 줄여가면서 최종 RMSE가 어떻게 바뀌는지**(학습 곡선).

In [8]:
from sklearn.datasets import load_diabetes
d = load_diabetes()
Xd, yd = d.data, d.target
Xs = (Xd - Xd.mean(0)) / Xd.std(0)
m_all = len(yd)
A_all = np.column_stack([np.ones(m_all), Xs])

# --- (a) 같은 실데이터(442명)에서 세 학습률의 비용 곡선 ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5))
for a, c, mk in ((0.01, "C0", "o"), (0.05, "C1", "s"), (0.2, "C2", "^")):
    w = np.zeros(11); hist = []
    for s in range(1, 6001):
        err = A_all @ w - yd
        w -= a * A_all.T @ err / m_all
        hist.append(0.5 / m_all * (err @ err))
        if s % 50 == 0 or s == 1:
            pass
    ax1.semilogy(range(1, len(hist) + 1), hist, color=c, lw=1.2, label=f"\u03b1={a}")
    print(f"alpha={a}: J[1]={hist[0]:.2f}  J[100]={hist[99]:.2f}  J[1000]={hist[999]:.2f}  J[6000]={hist[-1]:.2f}")
ax1.set_xlabel("Step"); ax1.set_ylabel("J (log)")
ax1.set_title("(a) diabetes (n=442, standardized) -- learning rate sets convergence speed")
ax1.legend(fontsize=8); ax1.grid(alpha=0.3, which="both")

# --- (b) 학습 데이터 크기 n에 따른 최종 RMSE (학습 곡선) ---
sizes = [50, 100, 250, 442]
rmse_list = []
for ntr in sizes:
    A = np.column_stack([np.ones(ntr), Xs[:ntr]])
    w = np.zeros(11)
    ws = np.linalg.solve(A.T @ A, A.T @ yd[:ntr])
    Jf = 0.5 / ntr * ((A @ ws - yd[:ntr]) ** 2).sum()
    for s in range(1, 100001):
        err = A @ w - yd[:ntr]
        w -= 0.05 * A.T @ err / ntr
        if abs(0.5 / ntr * (err @ err) - Jf) < 1e-3 * Jf:
            break
    rmse_all = np.sqrt(((A_all @ w - yd) ** 2).mean())
    rmse_list.append(rmse_all)
    print(f"n={ntr}: 스텝={s}  전체 442명 RMSE={rmse_all:.2f}")
ax2.plot(sizes, rmse_list, "s-", lw=2, color="C1")
ax2.set_xlabel("Training set size n"); ax2.set_ylabel("RMSE on all 442")
ax2.set_title("(b) learning curve -- more data, better model")
ax2.grid(alpha=0.3)
fig.suptitle("diabetes real data: (a) step size, (b) number of steps (=data) -- the two variables of gradient descent")
fig.tight_layout()
fig.savefig(f"{IMG}/ch02_diabetes_learning_curve.svg")
print("ch02_diabetes_learning_curve.svg 저장됨")
plt.show()

alpha=0.01: J[1]=14537.24  J[100]=3105.97  J[1000]=1439.36  J[6000]=1433.86
alpha=0.05: J[1]=14537.24  J[100]=1440.93  J[1000]=1434.61  J[6000]=1429.91
alpha=0.2: J[1]=14537.24  J[100]=1437.84  J[1000]=1430.21  J[6000]=1429.85
n=50: 스텝=2869  전체 442명 RMSE=58.72
n=100: 스텝=912  전체 442명 RMSE=56.33
n=250: 스텝=2807  전체 442명 RMSE=53.91
n=442: 스텝=2405  전체 442명 RMSE=53.50


ch02_diabetes_learning_curve.svg 저장됨


(a)에서는 \(\alpha=0.05\)보다 큰 \(\alpha=0.2\)도 수렴하지만(경계 0.50 안쪽), 비용이 더 빠르게 바닥에 붙는 것을 볼 수 있다 — §4의 "10배 규칙"이 실데이터에서도 대략 성립한다. (b)는 데이터 크기의 정반대 효과: \(n\)이 작으면 경사하강법은 **덜 걸어야** 하지만, 그 결과로 나오는 모델 자체가 나빠진다(RMSE 58.7 → 53.5). **학습 데이터가 442명 전체일 때의 RMSE 53.5가 이 데이터의 정규방정식(OLS) 해와 일치**한다 — 2.2절에서 "경사하강법과 정규방정식은 같은 목적지"라는 말이 실데이터 위에서 검증된 셈이다. 주의: (b)의 스텝 수가 n=100에서 갑자기 줄어드는 것처럼 보이는 것은 소음 바닥 자체가 n에 따라 달라지기 때문이지, "데이터가 적으면 빠르다"는 일반 법칙이 아니다.

## 8. 요약: 이 절의 숫자 한 장

| 항목 | 값 |
|---|---|
| \(X=[1,2,3]\), \(y=[3,5,7]\), 5스텝(\(\alpha=0.1\)) | \((0.889, 2.006)\), \(J=0.0049\) — 본문 표와 일치 |
| 그래디언트 수치미분 검증 | 50개 임의 점에서 최대 오차 \(<10^{-6}\) |
| 발산 경계값 \(\alpha^*=2/\lambda_{\max}\) (이 데이터) | \(\approx 0.3606\) → \(\alpha=0.5\)도 발산 |
| 300개 데이터, 바닥 도달 스텝 | \(\alpha=0.01\): 489, \(\alpha=0.1\): 47 (10배 규칙) |
| diabetes 표준화(442명) | \(\lambda_{\max} \approx 4.0\), 경계 \(\alpha^*\approx 0.50\) → \(\alpha=0.05\) 안정; 학습 곡선 RMSE 58.7(50)→53.5(442) = OLS 해 |

**핵심 교훈**: 경사하강법은 "수식 한 줄"이지만 실제로는 (1) 그래디언트가 정말 미분인지(§2), (2) 학습률이 데이터의 경계값 안에 있는지(§3), (3) 걸음 수가 충분한지(§4, §7)를 점검하는 세 단계의 검증 대상이다. 다음 2.2절에서는 같은 \(J(w)\)를 **한 번의 행렬 계산**으로 푸는 정규방정식, 그리고 그 골짜기의 "늘어짐"을 재는 **조건수**를 만난다.